# İlaç Tanıma Sistemi — Colab'da Model Eğitimi

Bu notebook, web uygulamasındaki **☁️ Drive & Colab** sekmesinden Google
Drive'a yüklediğiniz veri seti zip'ini kullanarak aynı 8 model mimarisini
Colab'ın (genelde ücretsiz) GPU'sunda eğitir.

**Adımlar:**
1. Çalışma zamanı türünü GPU yapın: `Çalışma zamanı > Çalışma zamanı türünü değiştir > GPU`.
2. Aşağıdaki hücreleri sırayla çalıştırın.
3. Eğitim bitince oluşan `model_output.zip` dosyasını indirin.
4. Web uygulamasındaki **☁️ Drive & Colab** sekmesinde "Eğitilen Modeli İçeri Al"
   formuyla bu zip'in içindeki `.h5` ve `.json` dosyalarını yükleyin.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Veri seti zip'ini bulun

Web uygulaması veri setini Drive'da **IlacTanimaVeriSeti** klasörüne
`dataset_YYYYMMDD_HHMMSS.zip` adıyla yükler. Aşağıdaki hücre bu klasördeki
en güncel zip dosyasını otomatik bulur; farklı bir dosya kullanmak isterseniz
`ZIP_PATH` değişkenini elle yazabilirsiniz.


In [ ]:
import glob
import os

DRIVE_FOLDER = "/content/drive/MyDrive/IlacTanimaVeriSeti"
zips = sorted(glob.glob(os.path.join(DRIVE_FOLDER, "dataset_*.zip")))
assert zips, f"{DRIVE_FOLDER} içinde dataset_*.zip bulunamadı. Önce web uygulamasından Drive'a yükleyin."

ZIP_PATH = zips[-1]  # en güncel zip
print("Kullanılacak zip:", ZIP_PATH)

EXTRACT_DIR = "/content/dataset_extracted"
OUTPUT_DIR = os.path.join(DRIVE_FOLDER, "model_output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

import zipfile
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(EXTRACT_DIR)

augmented = os.path.join(EXTRACT_DIR, "augmented_dataset")
raw = os.path.join(EXTRACT_DIR, "dataset")


def _has_at_least_2_classes(base_dir):
    if not os.path.isdir(base_dir):
        return False
    classes_with_images = [
        name for name in os.listdir(base_dir)
        if os.path.isdir(os.path.join(base_dir, name))
        and len(os.listdir(os.path.join(base_dir, name))) > 0
    ]
    return len(classes_with_images) >= 2


DATASET_DIR = augmented if _has_at_least_2_classes(augmented) else raw
print("Eğitim için kullanılacak klasör:", DATASET_DIR)


## Model tanımları

Aşağıdaki hücre `web/model_training.py` ile birebir aynı model mimarilerini
ve `train_all_models` fonksiyonunu içerir.


In [ ]:
import os
import json
import datetime

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, Model
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2

MODELS_CONFIG = {
    "basic_cnn": {"image_size": 96, "epochs": 15},
    "advanced_cnn": {"image_size": 96, "epochs": 20},
    "ensemble_cnn": {"image_size": 96, "epochs": 25},
    "vgg16": {"image_size": 224, "epochs": 10},
    "resnet50": {"image_size": 224, "epochs": 10},
    "mobilenet": {"image_size": 224, "epochs": 10},
    "efficientnet_fixed": {"image_size": 224, "epochs": 12},
    "vision_transformer_fixed": {"image_size": 96, "epochs": 25},
}


def create_basic_cnn(num_classes):
    return models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(96, 96, 3)),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D(2, 2),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax'),
    ])


def create_advanced_cnn(num_classes):
    return models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(96, 96, 3)),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax'),
    ])


def create_ensemble_cnn(num_classes):
    input_layer = layers.Input(shape=(96, 96, 3))

    x1 = layers.Conv2D(32, (3, 3), activation='relu')(input_layer)
    x1 = layers.MaxPooling2D(2, 2)(x1)
    x1 = layers.Conv2D(64, (3, 3), activation='relu')(x1)
    x1 = layers.GlobalAveragePooling2D()(x1)
    x1 = layers.Dense(64, activation='relu')(x1)

    x2 = layers.Conv2D(32, (5, 5), activation='relu')(input_layer)
    x2 = layers.MaxPooling2D(2, 2)(x2)
    x2 = layers.Conv2D(64, (5, 5), activation='relu')(x2)
    x2 = layers.GlobalAveragePooling2D()(x2)
    x2 = layers.Dense(64, activation='relu')(x2)

    x3 = layers.Conv2D(16, (3, 3), activation='relu')(input_layer)
    x3 = layers.Conv2D(32, (3, 3), activation='relu')(x3)
    x3 = layers.MaxPooling2D(2, 2)(x3)
    x3 = layers.Conv2D(64, (3, 3), activation='relu')(x3)
    x3 = layers.GlobalAveragePooling2D()(x3)
    x3 = layers.Dense(64, activation='relu')(x3)

    combined = layers.concatenate([x1, x2, x3])
    combined = layers.Dense(128, activation='relu')(combined)
    combined = layers.Dropout(0.5)(combined)
    output = layers.Dense(num_classes, activation='softmax')(combined)

    return Model(inputs=input_layer, outputs=output)


def create_vgg16_model(num_classes):
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    return models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax'),
    ])


def create_resnet50_model(num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    return models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax'),
    ])


def create_mobilenet_model(num_classes):
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    return models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax'),
    ])


def create_efficientnet_model(num_classes):
    try:
        from tensorflow.keras.applications import EfficientNetV2B0
        base_model = EfficientNetV2B0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    except Exception:
        base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base_model.trainable = False
    return models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation='softmax'),
    ])


class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, d_model, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size
        self.d_model = d_model

    def build(self, input_shape):
        self.patch_conv = layers.Conv2D(
            filters=self.d_model, kernel_size=self.patch_size,
            strides=self.patch_size, padding="valid", name="patch_conv",
        )
        super().build(input_shape)

    def call(self, images):
        patches = self.patch_conv(images)
        batch_size = tf.shape(patches)[0]
        num_patches = patches.shape[1] * patches.shape[2]
        return tf.reshape(patches, [batch_size, num_patches, self.d_model])

    def get_config(self):
        config = super().get_config()
        config.update({"patch_size": self.patch_size, "d_model": self.d_model})
        return config


def create_vision_transformer(num_classes, image_size=96, patch_size=16, num_layers=4, d_model=128, num_heads=4):
    inputs = layers.Input(shape=(image_size, image_size, 3))
    patches = PatchEmbedding(patch_size, d_model)(inputs)
    num_patches = (image_size // patch_size) ** 2

    position_embedding = layers.Embedding(input_dim=num_patches, output_dim=d_model, name="position_embedding")
    positions = tf.range(start=0, limit=num_patches, delta=1)
    encoded_patches = patches + position_embedding(positions)

    for i in range(num_layers):
        x1 = layers.LayerNormalization(epsilon=1e-6, name=f"norm1_{i}")(encoded_patches)
        attention_output = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model, name=f"attention_{i}",
        )(x1, x1)
        x2 = layers.Add(name=f"add1_{i}")([attention_output, encoded_patches])

        x3 = layers.LayerNormalization(epsilon=1e-6, name=f"norm2_{i}")(x2)
        x3 = layers.Dense(d_model * 2, activation="gelu", name=f"mlp_dense1_{i}")(x3)
        x3 = layers.Dense(d_model, name=f"mlp_dense2_{i}")(x3)
        encoded_patches = layers.Add(name=f"add2_{i}")([x3, x2])

    representation = layers.LayerNormalization(epsilon=1e-6, name="final_norm")(encoded_patches)
    representation = layers.GlobalAveragePooling1D(name="global_avg_pool")(representation)
    representation = layers.Dropout(0.3, name="final_dropout")(representation)
    outputs = layers.Dense(num_classes, activation="softmax", name="classifier")(representation)

    return Model(inputs=inputs, outputs=outputs, name="vision_transformer")


MODEL_BUILDERS = {
    "basic_cnn": lambda n: create_basic_cnn(n),
    "advanced_cnn": lambda n: create_advanced_cnn(n),
    "ensemble_cnn": lambda n: create_ensemble_cnn(n),
    "vgg16": lambda n: create_vgg16_model(n),
    "resnet50": lambda n: create_resnet50_model(n),
    "mobilenet": lambda n: create_mobilenet_model(n),
    "efficientnet_fixed": lambda n: create_efficientnet_model(n),
    "vision_transformer_fixed": lambda n: create_vision_transformer(n),
}


class _ProgressCallback(tf.keras.callbacks.Callback):
    def __init__(self, on_progress, model_name, total_epochs):
        super().__init__()
        self.on_progress = on_progress
        self.model_name = model_name
        self.total_epochs = total_epochs

    def on_epoch_end(self, epoch, logs=None):
        self.on_progress({
            "event": "epoch_end",
            "model": self.model_name,
            "epoch": epoch + 1,
            "total_epochs": self.total_epochs,
            "logs": {k: float(v) for k, v in (logs or {}).items()},
        })


def train_all_models(dataset_dir, output_dir, on_progress=None):
    """dataset_dir altındaki sınıf klasörlerinden tüm modelleri eğitir ve
    sonuçları output_dir içine (ilac_model_<isim>.h5, class_names*.json,
    model_results.json) yazar. on_progress(dict) her önemli adımda çağrılır."""

    def emit(update):
        if on_progress:
            on_progress(update)

    os.makedirs(output_dir, exist_ok=True)

    datagen = ImageDataGenerator(rescale=1. / 255, validation_split=0.2)

    gens = {}
    for size in (96, 224):
        gens[size] = {
            "train": datagen.flow_from_directory(
                dataset_dir, target_size=(size, size), batch_size=16,
                class_mode='categorical', subset='training',
            ),
            "val": datagen.flow_from_directory(
                dataset_dir, target_size=(size, size), batch_size=16,
                class_mode='categorical', subset='validation',
            ),
        }

    train_gen_96 = gens[96]["train"]
    num_classes = len(train_gen_96.class_indices)
    class_names = {v: k for k, v in train_gen_96.class_indices.items()}

    emit({"event": "start", "num_classes": num_classes, "classes": list(class_names.values()),
          "models": list(MODELS_CONFIG.keys())})

    results = {}
    for model_name, cfg in MODELS_CONFIG.items():
        train_gen = gens[cfg["image_size"]]["train"]
        val_gen = gens[cfg["image_size"]]["val"]
        emit({"event": "model_start", "model": model_name, "total_epochs": cfg["epochs"]})

        try:
            model = MODEL_BUILDERS[model_name](num_classes)
            model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

            history = model.fit(
                train_gen,
                validation_data=val_gen,
                epochs=cfg["epochs"],
                verbose=0,
                callbacks=[
                    tf.keras.callbacks.EarlyStopping(
                        monitor='val_accuracy', patience=5, restore_best_weights=True,
                    ),
                    _ProgressCallback(emit, model_name, cfg["epochs"]),
                ],
            )

            val_loss, val_acc = model.evaluate(val_gen, verbose=0)
            results[model_name] = {
                "val_loss": float(val_loss),
                "val_accuracy": float(val_acc),
                "params": int(model.count_params()),
                # Grafik çizmek için epoch bazlı accuracy/loss geçmişi.
                "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
            }

            model.save(os.path.join(output_dir, f"ilac_model_{model_name}.h5"))
            with open(os.path.join(output_dir, f"class_names_{model_name}.json"), "w", encoding="utf-8") as f:
                json.dump(class_names, f, ensure_ascii=False, indent=2)

            emit({"event": "model_end", "model": model_name, **results[model_name]})
        except Exception as e:
            results[model_name] = {"error": str(e)}
            emit({"event": "model_error", "model": model_name, "error": str(e)})

    valid_results = {k: v for k, v in results.items() if "error" not in v}
    best_model = None
    if valid_results:
        best_model = max(valid_results, key=lambda k: valid_results[k]["val_accuracy"])
        import shutil
        shutil.copy(
            os.path.join(output_dir, f"ilac_model_{best_model}.h5"),
            os.path.join(output_dir, "ilac_model.h5"),
        )
        shutil.copy(
            os.path.join(output_dir, f"class_names_{best_model}.json"),
            os.path.join(output_dir, "class_names.json"),
        )

    with open(os.path.join(output_dir, "model_results.json"), "w", encoding="utf-8") as f:
        json.dump(valid_results, f, ensure_ascii=False, indent=2)

    emit({
        "event": "done",
        "best_model": best_model,
        "results": valid_results,
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    })

    return results


## Eğitimi başlat

In [ ]:
def on_progress(update):
    event = update.get("event")
    if event == "model_start":
        print(f"\n=== {update['model']} eğitimi başladı ({update['total_epochs']} epoch) ===")
    elif event == "epoch_end":
        logs = update.get("logs", {})
        print(f"[{update['model']}] epoch {update['epoch']}/{update['total_epochs']} "
              f"- acc: {logs.get('accuracy', float('nan')):.4f} "
              f"- val_acc: {logs.get('val_accuracy', float('nan')):.4f}")
    elif event == "model_end":
        print(f"[{update['model']}] tamamlandı - val_accuracy: {update['val_accuracy']:.4f}")
    elif event == "model_error":
        print(f"[{update['model']}] HATA: {update['error']}")
    elif event == "done":
        print(f"\nEğitim tamamlandı. En iyi model: {update['best_model']}")


results = train_all_models(DATASET_DIR, OUTPUT_DIR, on_progress=on_progress)


## Eğitim sonuçları ve grafikler

Aşağıdaki hücreler eğitim biter bitmez modellerin karşılaştırmalı tablosunu
ve her model için accuracy/loss eğrilerini çizer. Grafikler `graphs/`
klasörüne PNG olarak da kaydedilir, böylece bir sonraki bölümdeki
`model_output.zip` indirmesine otomatik dahil olurlar.

In [ ]:
import pandas as pd

valid_results = {k: v for k, v in results.items() if "error" not in v}
failed = {k: v for k, v in results.items() if "error" in v}

summary = pd.DataFrame([
    {
        "model": name,
        "val_accuracy": r["val_accuracy"],
        "val_loss": r["val_loss"],
        "params": r["params"],
    }
    for name, r in valid_results.items()
]).sort_values("val_accuracy", ascending=False).reset_index(drop=True)

print(f"En iyi model: {summary.iloc[0]['model']} (val_accuracy={summary.iloc[0]['val_accuracy']:.4f})")
if failed:
    print(f"Eğitilemeyen modeller: {list(failed.keys())}")

summary

In [ ]:
import matplotlib.pyplot as plt

GRAPHS_DIR = os.path.join(OUTPUT_DIR, "graphs")
os.makedirs(GRAPHS_DIR, exist_ok=True)

# Her model için accuracy/loss eğrileri (eğitim vs doğrulama)
for model_name, r in valid_results.items():
    hist = r.get("history", {})
    if not hist:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(hist.get("accuracy", []), label="train")
    axes[0].plot(hist.get("val_accuracy", []), label="val")
    axes[0].set_title(f"{model_name} — Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(hist.get("loss", []), label="train")
    axes[1].plot(hist.get("val_loss", []), label="val")
    axes[1].set_title(f"{model_name} — Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(GRAPHS_DIR, f"{model_name}_curves.png"), dpi=120)
    plt.show()

# Modeller arası karşılaştırma
if valid_results:
    names = summary["model"].tolist()
    accs = summary["val_accuracy"].tolist()

    plt.figure(figsize=(10, 5))
    bars = plt.bar(names, accs, color="#4C72B0")
    plt.ylabel("Validation Accuracy")
    plt.title("Modeller Arası Karşılaştırma")
    plt.xticks(rotation=30, ha="right")
    plt.ylim(0, 1)
    for bar, acc in zip(bars, accs):
        plt.text(bar.get_x() + bar.get_width() / 2, acc + 0.01, f"{acc:.3f}", ha="center")
    plt.tight_layout()
    plt.savefig(os.path.join(GRAPHS_DIR, "model_comparison.png"), dpi=120)
    plt.show()

print(f"\nGrafikler kaydedildi: {GRAPHS_DIR}")

## Sonuçları indirin

Aşağıdaki hücre `OUTPUT_DIR` içindeki tüm `.h5` / `.json` dosyalarını **ve
yukarıda üretilen `graphs/` klasöründeki PNG grafikleri** tek bir
`model_output.zip` haline getirip indirir. Zip'i açtığınızda:
- `.h5` / `.json` dosyalarını web uygulamasının **☁️ Drive & Colab** sekmesindeki
  "Eğitilen Modeli İçeri Al" formuna sürükleyin,
- `graphs/` klasöründeki PNG'ler ise eğitim/karşılaştırma grafiklerinin
  kendisidir, doğrudan görüntüleyip paylaşabilirsiniz.

In [ ]:
import shutil
from google.colab import files

zip_base = "/content/model_output"
shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
files.download(zip_base + ".zip")
